# Landscape plotting with Matplotlib

Static views from `opfunu.plot.Plotting`: 2D contour maps and 3D surfaces of benchmark
landscapes, publication-ready exports (PDF, SVG, PGF) and custom styles.

```bash
pip install "opfunu-core[plot]"
```

Everything is lazy — the landscape is sampled and rendered on the first
`show()` / `save()`, and the grid is cached for every later call. Styles and rc
overrides apply per plot; global `rcParams` are never touched.

In [ ]:
from pathlib import Path

from opfunu.cec_based.cec2010 import F12010
from opfunu.plot import HAS_MATPLOTLIB, Plotting

print("matplotlib available:", HAS_MATPLOTLIB)

f0 = F12010(ndim=10)
figures = Path("figures")
figures.mkdir(exist_ok=True)

## 1. 2D contour map

`selected_dims` are 1-based; all other dimensions are fixed at their bound
midpoints. Evaluating `plot2d` as the last expression of a cell displays the
figure inline via its PNG repr — `plot2d.show()` does the same explicitly.

In [ ]:
plot2d = Plotting(
    f0.evaluate,                    # bounds inferred from f0.lb / f0.ub
    backend="matplotlib",
    mode="2d",
    selected_dims=(2, 3),
    n_points=200,
    cmap="viridis",
)
plot2d

## 2. 3D surface

`mode="3d"` renders the same grid as a surface (it is the default mode of
`Plotting` because it works for every backend).

In [ ]:
plot3d = Plotting(f0.evaluate, mode="3d", selected_dims=(1, 6), n_points=100)
plot3d.show()

## 3. Different slices and colormaps

Any pair of dimensions can be rendered and every backend accepts the usual
colormap names.

In [ ]:
Plotting(f0.evaluate, mode="2d", selected_dims=(4, 7), cmap="cividis", n_points=200).show()

## 4. Custom styles

`style` accepts a registered style name, a list of names, a path to a
`.mplstyle` file or a dict of rc parameters; `rc` layers overrides on top of
it. Styles from third-party packages work as well — import the package in the
notebook first so matplotlib knows the names. `styles/paper.mplstyle` ships
with this example folder.

In [ ]:
styled = Plotting(
    f0.evaluate,
    mode="2d",
    n_points=200,
    title="F12010 — paper.mplstyle",
    style="styles/paper.mplstyle",   # relative to this notebook's folder
    rc={"figure.dpi": 150},          # per-plot override on top of the style
)
styled.show()

In [ ]:
built_in = Plotting(f0.evaluate, mode="2d", n_points=200, style="ggplot")
built_in.show()

## 5. Publication exports

`save()` picks the format from the file suffix. Matplotlib covers raster
(`png`, `jpg`, `tif`, `webp`) and vector (`pdf`, `svg`, `svgz`, `eps`, `ps`,
`pgf`) formats, defaulting to `dpi=300` and `bbox_inches="tight"`.
`.pgf` emits LaTeX-native figure source for a paper and needs a TeX
installation.

In [ ]:
styled.save(figures / "landscape2d.pdf")
styled.save(figures / "landscape2d.svg")
styled.save(figures / "landscape2d.png", dpi=600)
# LaTeX-native vector output; requires a TeX installation:
styled.save(figures / "landscape2d.pgf")

## 6. Native figure access

`plot.figure` exposes the underlying matplotlib `Figure` for direct tweaks —
then `save()` writes the customized result.

In [ ]:
fig = plot2d.figure
fig.axes[0].set_title("Custom title on the native matplotlib figure")
plot2d.save(figures / "landscape2d_custom.pdf")

## 7. Cleanup

`close()` releases the native figure; the sampled grid stays cached, so a
later `render()` rebuilds the figure cheaply.

In [ ]:
plot2d.close()
plot3d.close()
styled.close()